In [7]:
import xarray as xr
import rioxarray
import gcsfs

# Load service account key
fs = gcsfs.GCSFileSystem(token=r"C:\Users\Adriano Matos\Documents\credentials\cloud_storage\robust-raster-562c48f78248.json")

# Sample xarray dataset
ds = xr.Dataset(
    {
        "band1": (("y", "x"), [[15, 20], [25, 30]]),
    },
    coords={
        "x": [0, 1],
        "y": [0, 1],
    },
)

# Convert to DataArray and set CRS
da = ds["band1"]
da = da.rio.write_crs("EPSG:4326")  # Set CRS

import rasterio
from rasterio.io import MemoryFile
import gcsfs

# Load Google Cloud Storage filesystem with service account
fs = gcsfs.GCSFileSystem(token=r"C:\Users\Adriano Matos\Documents\credentials\cloud_storage\robust-raster-562c48f78248.json")

# Define GCS path
gcs_path = "gcs://test-xarrgcs-bucket/output_cog.tif"

# Create in-memory file
with MemoryFile() as memfile:
    with memfile.open(
        driver="COG",
        width=da.rio.width,
        height=da.rio.height,
        count=1,
        dtype=da.dtype,
        crs=da.rio.crs,
        transform=da.rio.transform(),
    ) as dataset:
        dataset.write(da.values, 1)  # Write the data into the memory file

    # Open GCS path and write the MemoryFile contents
    with fs.open(gcs_path, "wb") as f:
        f.write(memfile.read())

print("COG successfully written to GCS!")


COG successfully written to GCS!
